# 3. Invalid passage references

Here the JSON and schema pass, but at least one cited passage ID is absent from the supplied paper passages.

Run cells from top to bottom. Everything runs offline; no API key or model request is used.

In [ ]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Works when opened from the repository, notebooks/, or its parent directory.
root = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "research_finder"]
            if (p / "novelty/comparison_workflow.py").exists())
sys.path.insert(0, str(root))
# Check the selected kernel before importing project modules.
import importlib.util
missing = [name for name in ("openai", "pydantic", "pymupdf")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Kernel {sys.executable} is missing: {', '.join(missing)}. "
        f"In VS Code choose Select Kernel > Python Environments > {root / '.venv/bin/python'}, "
        "then restart the kernel and Run All."
    )
from notebooks.example_helpers import load_example, expect_failure
from extraction.research_extract import _parse_json
from novelty.comparison_records import EvidenceReport, EvidenceRecord, EvidenceReview
from novelty.comparison_schemas import Passage
from novelty.comparison_workflow import validate_evidence_report, validate_record, eligible, scope_citation_gaps
from novelty.schemas import TargetSignature


In [ ]:
example = load_example(root, "03_invalid_passage_references")
pprint(example["provenance"])

In [ ]:
raw = example["invalid_output"]
report = EvidenceReport.model_validate(_parse_json(raw) if isinstance(raw, str) else raw)
record = EvidenceRecord.model_validate(example["record"])
passages = [Passage.model_validate(p) for p in example["passages"]]
print("JSON and schema checks passed.")

In [ ]:
error = expect_failure(lambda: validate_evidence_report(report, record, passages, require_scope=True), "unknown/wrong-paper")
known = {p.passage_id for p in passages}
references = {pid for check in report.claim_checks + report.coverage_checks for pid in check.passage_ids}
references |= {pid for claim in report.claim_checks for check in claim.scope_checks or [] for pid in check.passage_ids}
print("Unknown references:")
pprint(sorted(references - known))
print("Example available passage:")
pprint(passages[0].model_dump())

An ID-shaped string is not enough: it must identify an actual supplied passage. This prevents nonexistent or wrong-paper evidence from being accepted. A real passage ID alone still does not prove that the text supports the claim.